> 在线讲义：[第 42 章 · 共享网络中的拥塞与可靠性：高峰、反馈、多路径、死锁](https://wikieden.github.io/ai-infra-from-constraints/lessons/42-congestion.html) · 本书《从约束推导 AI Infra》

# 第 42 章 · 共享网络中的拥塞与可靠性：高峰、反馈、多路径、死锁

配套讲义：仓库里 `ai-infra/notes/lessons/42-congestion.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



例题 7.5、反馈时延与缓冲、incast 与 PFC、多路径与重传、ECMP 哈希冲突、逐包喷洒取自 AII 第 7 章 §7.5。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
from fractions import Fraction
from functools import lru_cache
from math import comb, ceil
KiB, MiB = 2**10, 2**20

**函数 `overlap_backlog_gb`**：小工具。

In [ ]:
def overlap_backlog_gb(overlap_ms: float, peak: float = 50, out: float = 50) -> float:
    return (2 * peak - out) * overlap_ms / 1e3

**函数 `tolerable_delay_us`**：小工具。

In [ ]:
def tolerable_delay_us(free: float, arrive: float, out: float) -> float:
    return free / (arrive - out) * 1e6

**函数 `incast`**：(允许反馈时延 μs, PFC headroom B, 端到端 RTT 内超额 B)。

In [ ]:
def incast(N: int, free: float = MiB, B: float = 50e9, hop_s: float = 0.33e-6, rtt: float = 20e-6) -> tuple[float, float, float]:
    """(允许反馈时延 μs, PFC headroom B, 端到端 RTT 内超额 B)。"""
    excess = (N - 2) * B
    return free / excess * 1e6, excess * hop_s, excess * rtt

**函数 `pfc_hop_us`**：小工具。

In [ ]:
def pfc_hop_us(cable_m: float = 30, mtu: int = 1500, B: float = 50e9) -> float:
    return (2 * cable_m * 5e-9 + mtu / B) * 1e6

**函数 `multipath_us`**：小工具。

In [ ]:
def multipath_us(pkts: int = 8, size: int = 4096, delays: tuple[float, ...] = (1, 1), B: float = 50e9) -> float:
    per = size / B * 1e6
    n = len(delays)
    k = ceil(pkts / n)
    return max(d + k * per for d in delays)

**函数 `max_load_dist`**：n 个球独立均匀放进 m 个盒子，最大负载的精确分布。

In [ ]:
def max_load_dist(n: int, m: int) -> dict[int, Fraction]:
    """n 个球独立均匀放进 m 个盒子，最大负载的精确分布。"""
    @lru_cache(None)
    def ways(balls: int, boxes: int, cap: int) -> int:
        if boxes == 0:
            return 1 if balls == 0 else 0
        return sum(comb(balls, j) * ways(balls - j, boxes - 1, cap) for j in range(0, min(cap, balls) + 1))
    total = m**n
    cdf = {c: Fraction(ways(n, m, c), total) for c in range(0, n + 1)}
    return {c: cdf[c] - (cdf[c - 1] if c > 0 else 0) for c in range(1, n + 1)}

**函数 `ecmp`**：(E[Lmax], p99, 无冲突概率, 相对无冲突速度)。

In [ ]:
def ecmp(n: int, m: int) -> tuple[float, int, float, float]:
    """(E[Lmax], p99, 无冲突概率, 相对无冲突速度)。"""
    d = max_load_dist(n, m)
    e = sum(c * p for c, p in d.items())
    acc, p99 = Fraction(0), None
    for c in sorted(d):
        acc += d[c]
        if p99 is None and acc >= Fraction(99, 100):
            p99 = c
    ideal = ceil(n / m)
    return float(e), p99, float(d.get(ideal, 0)), ideal / float(e)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：例题 7.5")
print(f"  平均 {2 * 50 * 0.2:.0f} GB/s；重叠 20 ms 积压 {overlap_backlog_gb(20):.1f} GB，排空 {overlap_backlog_gb(20) / 50 * 1e3:.0f} ms；重叠 5 ms {overlap_backlog_gb(5) * 1e3:.0f} MB")
print(f"  512 KiB 缓冲容忍的重叠 {512 * KiB / 50e9 * 1e6:.1f} μs")

In [ ]:
print("\n第二步：反馈时延")
print(f"  剩 256 KiB、超额 50 GB/s：{tolerable_delay_us(256 * KiB, 100e9, 50e9):.1f} μs；20 μs 才降速丢 {(50e9 * 20e-6 - 256 * KiB) / 1e3:.0f} KB")
print(f"  降到 40 GB/s 排空 512 KiB {512 * KiB / 10e9 * 1e6:.0f} μs")
print(f"  PFC 一跳 {pfc_hop_us():.2f} μs")
for N in (8, 16, 64):
    t, h, e = incast(N)
    print(f"  incast N = {N:2d}：允许 {t:.2f} μs，headroom {h / 1e3:.0f} KB，20 μs 内超额 {e / 1e6:.1f} MB")

In [ ]:
print("\n第三步：多路径与重传（8 × 4 KiB）")
print(f"  两条 1 μs：{multipath_us():.2f}；单路径：{multipath_us(delays=(1,)):.2f}；第二条 9 μs：{multipath_us(delays=(1, 9)):.2f} μs")
print(f"  首包丢失 20 μs 后重传：{4096 / 50e9 * 1e6 + 20 + 4096 / 50e9 * 1e6 + 1:.1f} μs")
for n, m in ((8, 32), (8, 16), (32, 32), (32, 16), (128, 16)):
    e, p99, p0, sp = ecmp(n, m)
    print(f"  ECMP {n:3d} 流 → {m} 上联：E[Lmax] {e:.2f}，p99 {p99}，无冲突 {p0:.1%}，速度 {sp:.1%}")
spray = 1 * MiB / 50e9 * 1e6 + 8
print(f"  8 MiB 喷洒到 8 条路径（1–8 μs）：{spray:.1f} μs；单路径 {8 * MiB / 50e9 * 1e6 + 1:.1f} μs")

## 3. 代码里才看得到的

1. `max_load_dist` 用 `Fraction` 和组合数精确计数（$m^n$ 种放法），不是蒙特卡洛；128 球 16 盒也只要一两秒。
2. incast 的超额是 $(N - 2) \times 50$：$N - 1$ 个发送方减去出口能送走的一份。
3. 多路径模型里每条路径的完成 = 时延 + 分到的报文数 × 82 ns，取最大。

## 4. 自己改一改

- 用 `ecmp` 画 $n$ 从 1 到 64、$m = 32$ 的速度曲线，找最低点。
- 给 `multipath_us` 加不均匀分配，找时延 1 μs 和 9 μs 两条路径的最优分法。
- 模拟一个周期性 incast：每 100 μs 来一次 $N = 16$，看 PFC 暂停占了多少时间。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 两家都九点上班，路平时空也会堵；错开二十分钟就不堵；
> 站口举牌管一个路口、快，打电话管源头、慢；
> 两人分送八页，绕远路的拖全书；随机分道最怕车道和车一样多；
> 窄桥对开都等对方退，先后规矩或留一条退路。